# Second Order Optimality Conditions

```{warning}
**AI-drafted prose, not yet reviewed by Prof. Dowling.**

Some of the writing on this page was drafted or edited by an AI assistant and has not yet been reviewed: 1 rewritten markdown cells, measured against the last version of this notebook predating AI editing (2026-08-17).

This notice is about the *prose only*. It says nothing either way about the code, the numbers or the figures, which are checked separately. It is removed once the page has been reviewed.
```

In [1]:
# This code cell installs packages on Colab

import sys

if "google.colab" in sys.modules:
    !wget "https://raw.githubusercontent.com/ndcbe/optimization/main/notebooks/helper.py"
    import helper

    helper.easy_install()
else:
    sys.path.insert(0, "../")
    import helper
helper.set_plotting_style()

## Helpful Cones

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/cone1.png)

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/cone2.png)

## Second Order Necessary Conditions

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/thm-4-17.png)

## Second Order Sufficient Conditions

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/thm-4-18.png)

## Reduced Hessian

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/def-4-19.png)

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/projected-reduced-Hessian1.png)

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/projected-reduced-Hessian2.png)

## Example

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/ex-4-20.png)

See errata here: http://numero.cheme.cmu.edu/content/errata2.pdf

![picture](https://raw.githubusercontent.com/ndcbe/optimization/main/media/errata_83.png)

### Calculation with numpy

In [2]:
import numpy as np

# Define the Hessian of the Lagrange function
hessian_L = np.array([[2.3093, 0.4315], [0.4315, 3.2021]])
print(hessian_L)

[[2.3093 0.4315]
 [0.4315 3.2021]]


In [3]:
# Define the gradient of the constraints
A = np.reshape(np.array([-2.2245, -1.7981]), (2, 1))

In [4]:
print(A)

[[-2.2245]
 [-1.7981]]


Calculate the complete QR factorization.

$$
A = Q \times R
$$

In [5]:
# Calculate the COMPLETE QR factorization
Q, R = np.linalg.qr(A, mode="complete")

In [6]:
print(Q)

[[-0.77770385 -0.62863083]
 [-0.62863083  0.77770385]]


In [7]:
print(R)

[[2.86034331]
 [0.        ]]


In [8]:
print(Q @ R)

[[-2.2245]
 [-1.7981]]


The second column of $Q$ is the null space of $A$. How do we know it is the null space? It corresponds to the second element of $R$ which is 0.

Finally, we can calculate the reduced Hessian:

In [9]:
# Calculate the reduced Hessian
print("Reduced Hessian w.r.t. null space of constraints:")
Q[:, 1].T @ hessian_L @ Q[:, 1]

Reduced Hessian w.r.t. null space of constraints:


np.float64(2.4273753426093774)

### Calculate with Pyomo

#### Define and solve the model

In [10]:
import pyomo.environ as pyo

m = pyo.ConcreteModel()

# Define variables
m.x1 = pyo.Var(initialize=1)
m.x2 = pyo.Var(initialize=1)

# Define constraints
# Note: changing this to an equality constraint
# changes the reduced_hessian
m.con1 = pyo.Constraint(expr=4 - m.x1 * m.x2 <= 0)

# Define objective
m.obj = pyo.Objective(
    expr=m.x1**2
    - 4 * m.x1
    + 1.5 * m.x2**2
    - 7 * m.x2
    + m.x1 * m.x2
    + 9
    - pyo.log(m.x1)
    - pyo.log(m.x2)
)

# Obtain dual solutions from first solve and send to warm start
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

# Specify Ipopt as the solver and solve
opt = pyo.SolverFactory("ipopt")
results = opt.solve(m, tee=True)
assert pyo.check_optimal_termination(results), (
    f"Solve failed: status={results.solver.status}, "
    f"termination={results.solver.termination_condition}"
)

Ipopt 3.13.2: 

******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        computation. See http://

#### Extract the dual variable for the constraint

In [11]:
m.dual[m.con1]

-0.5684977067847851

This is negative because of a sign convention in Pyomo. Notice it matches the book (except the sign).

#### Extract the reduced Hessian

In [12]:
# Warning: this Pyomo feature is experimental

from pyomo.contrib.interior_point.inverse_reduced_hessian import (
    inv_reduced_hessian_barrier,
)

# https://github.com/Pyomo/pyomo/blob/main/pyomo/contrib/interior_point/inverse_reduced_hessian.py

Compute the reduced Hessian with respect to $x_1$

In [13]:
solve_result, inv_red_hes = inv_reduced_hessian_barrier(
    m,
    independent_variables=[m.x1],  # Warning: these variables cannot be at their bounds
    tee=True,
)

print("\nReduced Hessian w.r.t. x1:")
print(np.linalg.inv(inv_red_hes))

Ipopt 3.13.2: bound_relax_factor=0
honor_original_bounds=no


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-sca

Compute the reduced Hessian with respect to $x_2$

In [14]:
solve_result, inv_red_hes = inv_reduced_hessian_barrier(
    m,
    independent_variables=[m.x2],  # Warning: these variables cannot be at their bounds
    tee=True,
)

print("\nReduced Hessian w.r.t. x2:")
print(np.linalg.inv(inv_red_hes))

Ipopt 3.13.2: bound_relax_factor=0
honor_original_bounds=no


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-sca

**Take away message**: The reduced Hessian relative to all three bases (null space, $[1, 0]$, and $[0, 1]$) is positive definite.